In [1]:
print('hel;lo world')

hel;lo world


In [2]:
import pandas as pd
import xgboost as xgb
import numpy as np
import sklearn
import optuna

print("XGBoost Version", xgb.__version__)
print("Pandas Version", pd.__version__)
print("Numpy Version", np.__version__)
print("Sklearn Version", sklearn.__version__)
print("Optuna Version", optuna.__version__)

XGBoost Version 3.2.0
Pandas Version 3.0.3
Numpy Version 2.4.6
Sklearn Version 1.8.0
Optuna Version 4.8.0


d:\Coding\hyperparam_tuning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
df = pd.read_csv("life.csv")
df = df.dropna(subset=["Life expectancy "]).copy()
df.head()

,Country,Year,Status,Life expectancy,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,...,Polio,Total expenditure,Diphtheria,HIV/AIDS,GDP,Population,thinness 1-19 years,thinness 5-9 years,Income composition of resources,Schooling
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,1154,...,6.0,8.16,65.0,0.1,584.259210,33736494.0,17.2,17.3,0.479,10.1
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,492,...,58.0,8.18,62.0,0.1,612.696514,327582.0,17.5,17.5,0.476,10.0
2,Afghanistan,2013,Developing,59.9,268.0,66,0.01,73.219243,64.0,430,...,62.0,8.13,64.0,0.1,631.744976,31731688.0,17.7,17.7,0.470,9.9
3,Afghanistan,2012,Developing,59.5,272.0,69,0.01,78.184215,67.0,2787,...,67.0,8.52,67.0,0.1,669.959000,3696958.0,17.9,18.0,0.463,9.8
4,Afghanistan,2011,Developing,59.2,275.0,71,0.01,7.097109,68.0,3013,...,68.0,7.87,68.0,0.1,63.537231,2978599.0,18.2,18.2,0.454,9.5


In [12]:
COLS_TO_DROP = ["Year", "Status", "Country", "Life expectancy "]

X = df[[c for c in df.columns if c not in COLS_TO_DROP]]
y = df["Life expectancy "]

In [25]:
import xgboost as xgb
from sklearn.model_selection import GroupKFold, cross_validate
import optuna


# Defining the objective function
def objective_gpu(trial):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
    }

    model = xgb.XGBRegressor(**param, device="cuda")

    scores = cross_validate(
        model,
        X,
        y,
        cv=GroupKFold(n_splits=3),
        groups=df["Country"],
        scoring={
            "mae": "neg_mean_absolute_error",
            "mape": "neg_mean_absolute_percentage_error",
        },
        error_score="raise",
    )

    fold_mapes = -scores["test_mape"] * 100
    print(f"Fold MAPE (%): {fold_mapes}")
    print(f"Mean MAPE (%): {fold_mapes.mean():.2f}")

    return scores["test_mae"].mean()

In [26]:
# Create and run the optimization process with 10 trials
study = optuna.create_study(study_name="xgboost_study_cuda", direction="maximize")
study.optimize(objective_gpu, n_trials=10, show_progress_bar=True, n_jobs=-1)

# Retrieve the best parameter values
best_params = study.best_params
print(f"\nBest parameters: {best_params}")

[I 2026-05-25 21:53:56,123] A new study created in memory with name: xgboost_study_cuda
Best trial: 3. Best value: -2.01026:  10%|█         | 1/10 [00:12<01:55, 12.80s/it]

Fold MAPE (%): [3.14940471 3.04989978 2.95129676]
Mean MAPE (%): 3.05
[I 2026-05-25 21:54:08,923] Trial 3 finished with value: -2.01025716203158 and parameters: {'max_depth': 4, 'learning_rate': 0.09949162732057562, 'n_estimators': 249, 'subsample': 0.9689119628017728, 'colsample_bytree': 0.9028306011224054, 'min_child_weight': 4, 'gamma': 3.447752965395241}. Best is trial 3 with value: -2.01025716203158.


Best trial: 3. Best value: -2.01026:  20%|██        | 2/10 [00:13<00:45,  5.69s/it]

Fold MAPE (%): [3.15515306 3.12499727 3.00632784]
Mean MAPE (%): 3.10
[I 2026-05-25 21:54:09,635] Trial 6 finished with value: -2.046488250669886 and parameters: {'max_depth': 3, 'learning_rate': 0.04820480426001005, 'n_estimators': 248, 'subsample': 0.6708931319612549, 'colsample_bytree': 0.721920383049072, 'min_child_weight': 2, 'gamma': 0.7608541328738994}. Best is trial 3 with value: -2.01025716203158.


Best trial: 2. Best value: -1.9829:  30%|███       | 3/10 [00:18<00:36,  5.15s/it] 

...

Fold MAPE (%): [3.37107853 3.18025204 2.80321562]
Mean MAPE (%): 3.12
[I 2026-05-25 21:54:20,738] Trial 0 finished with value: -2.047170332872151 and parameters: {'max_depth': 10, 'learning_rate': 0.03540616317849148, 'n_estimators': 186, 'subsample': 0.6535794758688347, 'colsample_bytree': 0.90663246885412, 'min_child_weight': 6, 'gamma': 0.4066936286123368}. Best is trial 1 with value: -1.979702890114706.


Best trial: 1. Best value: -1.9797:  90%|█████████ | 9/10 [00:25<00:01,  1.10s/it]

Fold MAPE (%): [3.37866082 3.15403061 2.83715944]
Mean MAPE (%): 3.12
[I 2026-05-25 21:54:21,363] Trial 4 finished with value: -2.0512005334343417 and parameters: {'max_depth': 7, 'learning_rate': 0.0893063367885543, 'n_estimators': 835, 'subsample': 0.9720270352855938, 'colsample_bytree': 0.6624218857436028, 'min_child_weight': 5, 'gamma': 2.6277101031252545}. Best is trial 1 with value: -1.979702890114706.


Best trial: 1. Best value: -1.9797: 100%|██████████| 10/10 [00:26<00:00,  2.61s/it]

Fold MAPE (%): [3.27846838 3.11412182 2.99798422]
Mean MAPE (%): 3.13
[I 2026-05-25 21:54:22,243] Trial 7 finished with value: -2.061345745305546 and parameters: {'max_depth': 3, 'learning_rate': 0.04362715192194, 'n_estimators': 865, 'subsample': 0.8923356824669639, 'colsample_bytree': 0.6879423844192698, 'min_child_weight': 10, 'gamma': 3.980111888153415}. Best is trial 1 with value: -1.979702890114706.

Best parameters: {'max_depth': 5, 'learning_rate': 0.06248353827615622, 'n_estimators': 639, 'subsample': 0.8660474681447238, 'colsample_bytree': 0.9533428473013832, 'min_child_weight': 7, 'gamma': 4.026081602811956}


In [27]:
study.best_value

-1.979702890114706

In [28]:
best_model = xgb.XGBRegressor(**best_params, device="cuda")
best_model.fit(X, y)
predictions = best_model.predict(X)
predictions[40:2000:200], y[40:2000:200], predictions.shape, y.shape

(array([73.19332 , 82.76336 , 51.295166, 78.01162 , 55.538475, 73.9214  ,
        66.47678 , 75.02027 , 79.97423 , 81.778946], dtype=float32),
 40      73.8
 240     81.1
 440     49.9
 641     78.0
 842     55.0
 1042    73.6
 1242    65.9
 1442    74.6
 1642    79.6
 1845    81.6
 Name: Life expectancy , dtype: float64,
 (2928,),
 (2928,))